In [1]:
import os
import qdrant_client


from docling.datamodel.document import DoclingDocument

from llama_index.core.schema import TextNode
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes, MarkdownNodeParser
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    Settings,
    Document,
)

import shutil

from qdrant_client.models import (
    VectorParams,
    Distance,
    SparseVectorParams,
)

import importlib
import utils as utils
utils = importlib.reload(utils)
from utils import iniezionetagimmagini, riassuntodocumento, metadata_extraction
import sqlite3


In [2]:
collectionname = "WAMASHIERARCHICAL"

url_embedder = os.getenv("VLLM_API_BASE_URL")
url_qdrant = os.getenv("QDRANT_URL")

client = qdrant_client.QdrantClient(url=url_qdrant)

embed_model = OpenAIEmbedding(
    api_base=url_embedder,
    model_name="BAAI/bge-m3",
    api_key="null",
)
Settings.embed_model = embed_model

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collectionname,
    enable_hybrid=True,
    dense_vector_name="bge_m3",
    sparse_vector_name="bm25",
    fastembed_sparse_model="Qdrant/bm25"
)

docstore = SimpleDocumentStore()
storage_context = StorageContext.from_defaults(vector_store=vector_store, docstore=docstore)


In [3]:
if os.path.exists(f"../{collectionname}.db"):
    os.remove(f"../{collectionname}.db")

conn = sqlite3.connect(f"../{collectionname}.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS document_chunks (
        filename TEXT,
        chunk_index INTEGER,
        text_content TEXT,
        PRIMARY KEY (filename, chunk_index)
    )
""")
conn.commit()

In [4]:
if client.collection_exists(collectionname):
    client.delete_collection(collectionname)

client.create_collection(
    collection_name=collectionname,
    vectors_config={
        "bge_m3": VectorParams(
            size=1024,
            distance=Distance.COSINE
        )
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams()
    }
)

True

In [5]:
def qdrant_embedding_hierarchical_automerge(DOC_SOURCE, DOC_SOURCE_MD):
    # come prima qua
    doc = DoclingDocument.load_from_json(DOC_SOURCE)
    doc = iniezionetagimmagini(doc)
    full_text = doc.export_to_markdown()
    full_text = full_text.replace("<!-- image -->", "")
    DOC_SOURCE = DOC_SOURCE.split("/")[-1].replace(".json", ".pdf")
    gruppi = DOC_SOURCE.split("_")[0]
    gruppi = gruppi.split("-") if "-" in gruppi else [gruppi]

    document = [Document(text=full_text, metadata={"origin_filename": DOC_SOURCE, "groups": gruppi})]


    
    # 2048 (nonno) - 512 (padre) - 128 (foglia)
    node_parser = HierarchicalNodeParser.from_defaults(
        chunk_sizes=[2048, 512, 128]
    )
    
    
    all_nodes = node_parser.get_nodes_from_documents(document)
    
    # prendo solo le foglie
    leaf_nodes = get_leaf_nodes(all_nodes)

    sql_data = []

    for i,node in enumerate(leaf_nodes):
        node.metadata["chunk_index"] = i
        sql_data.append((node.metadata["origin_filename"], i, node.get_content()))

    cursor.executemany(
        "INSERT OR REPLACE INTO document_chunks (filename, chunk_index, text_content) VALUES (?, ?, ?)",
        sql_data
    )
    conn.commit()


    storage_context.docstore.add_documents(all_nodes)

    #creazione indice
    VectorStoreIndex(
        leaf_nodes,
        storage_context=storage_context,#riga importantissima (perso 1 ora)
        show_progress=True
    )

    # usare mongodb qua
    storage_context.persist(persist_dir="./storage_hierarchical")
    
    print(f"Indicizzati {len(leaf_nodes)} nodi foglia. Totale nodi gerarchici salvati: {len(all_nodes)}")

In [6]:
base_folder = "../preprocessing/scratch"

for root, dirs, files in os.walk(base_folder):
    json_file = None
    md_file = None

    for file in files:
        if file.endswith(".json"):
            json_file = os.path.join(root, file)
        elif file.endswith(".md"):
            md_file = os.path.join(root, file)

    if json_file and md_file:
        qdrant_embedding_hierarchical_automerge(
            DOC_SOURCE=json_file,
            DOC_SOURCE_MD=md_file
        )


Generating embeddings:   0%|          | 0/29 [00:00<?, ?it/s]

Indicizzati 29 nodi foglia. Totale nodi gerarchici salvati: 38


Generating embeddings:   0%|          | 0/26 [00:00<?, ?it/s]

Indicizzati 26 nodi foglia. Totale nodi gerarchici salvati: 34


Generating embeddings:   0%|          | 0/17 [00:00<?, ?it/s]

...

Indicizzati 31 nodi foglia. Totale nodi gerarchici salvati: 39


Generating embeddings:   0%|          | 0/15 [00:00<?, ?it/s]

Indicizzati 15 nodi foglia. Totale nodi gerarchici salvati: 19


Generating embeddings:   0%|          | 0/44 [00:00<?, ?it/s]

Indicizzati 44 nodi foglia. Totale nodi gerarchici salvati: 56
